In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
team_name="team_lemma"
catalog_name=f"charles_schwab_retailbrokerage_dev_{team_name}"
dbutils.widgets.text("batch_id","1","BATCH ID")
bronze_batchdate = f"{catalog_name}.bronze.batchdate"
silver_batchdate = f"{catalog_name}.silver.batchdate"

In [0]:
batch_id=dbutils.widgets.get("batch_id")

In [0]:
df_bronze=spark.read.table(bronze_batchdate)


In [0]:
df_silver=df_bronze\
    .withColumn("batchdate", to_date(col("batchdate"), "yyyy-MM-dd")) \
    .withColumn("batchid", col("_batch").cast("INT")) \
    .withColumn("_load_ts", current_timestamp()) \
    .drop("_ingest_ts", "_source_file")

In [0]:
df_silver.createOrReplaceTempView("df_silver")

df_dedup = spark.sql("""
WITH ranked_data AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY batchid
               ORDER BY _load_ts DESC
           ) AS rn
    FROM df_silver
)
SELECT * EXCEPT (rn)
FROM ranked_data
WHERE rn = 1
""")


In [0]:
try:
    print(f"Writing to {silver_batchdate}...")
    df_dedup.write.format("delta").mode("overwrite").saveAsTable(silver_batchdate)
    print(f"Successfully processed silver.batchdate. Total rows: {df_dedup.count()}")
except Exception as e:
    print("Error writing to table:", e)
    raise e

In [0]:


# log_pipeline_recon(
#     spark=spark,
#     run_id=carried_run_id,
#     batch_id="ALL",
#     domain="CONTROL",
#     table_name="batchdate",
#     source_layer="bronze",
#     target_layer="silver",
#     source_count=source_count,
#     target_count=target_count
# )

# log_audit_event(
#     spark=spark,
#     run_id=carried_run_id,
#     batch="ALL",
#     layer="silver",
#     table_name="batchdate",
#     operation="OVERWRITE",
#     rows_affected=target_count
# )